In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

In [12]:
os.environ['HF_HOME'] = 'D:/huggingface_cache'

In [13]:
load_dotenv('../.env')

True

## Indexing (Document Ingestion)

In [14]:
video_id = "MiUHjLxm3V0" # only the ID, not full URL
try:
    # If you don't care which language, this returns the "best" one
    transcript_list = YouTubeTranscriptApi().fetch(video_id=video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

- This is a microchip. When you zoom in, you find
a nanoscopic computing city, skyscrapers hundreds of layers tall with hundreds of kilometers of
wires connecting everything. And at the very bottom is this, transistors, billions of them. They are the ones and
zeros of our computer. The chip works by whizzing electrons from transistor to transistor, and the smaller you can
make those transistors, the less the signals have to travel, so the faster they can compute. Plus, you can fit more
transistors into the same area, resulting in a much more powerful chip. So for over 50 years, transistors
got smaller and smaller, and the number you could
fit on a chip doubled every two years. This became known as Moore's Law, named for Intel's
co-founder, Gordon Moore, after he noticed the pattern back in 1965, and it's been one of the main
drivers of the tech industry. But around 2015, progress came to a screeching halt, and we might have never gotten past it if it wasn't for a single company
that ma

## Indexing (Text Splitting)

In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks=splitter.create_documents([transcript])

In [16]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
)
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [17]:
vector_store.index_to_docstore_id

{0: '94507ebb-75d8-4112-a7b1-7ba435729db1',
 1: 'e035f871-4565-4368-a5d1-0fef41b9c2b8',
 2: 'f60f7a15-d27e-4707-ab71-c7732fd9cb9a',
 3: '5b4cc47b-85d5-4a6a-b867-226fcefe3463',
 4: 'ee276a65-023a-4d28-84e5-bb3f180529ec',
 5: 'd37aa296-d64b-4738-9f58-a8bffe40da95',
 6: '15d01e85-5ddd-4729-9cdb-fd272725734f',
 7: 'af2038a3-2f99-4aac-ae91-e6c048ab9e10',
 8: '7cb71ecf-88da-48bb-bdf8-cd6ee33a50be',
 9: 'bd402cf8-2181-4eee-81d5-b860d30d187a',
 10: '40cefa21-31f4-421c-9f82-560df226dc81',
 11: 'ee7c4994-2cab-46b1-ba0c-5e9bc68f6bf4',
 12: 'f9d270ae-83dd-409f-ad5d-435063a33855',
 13: 'e46565f9-fed2-4779-a146-194c0558fe9b',
 14: 'bec68de2-c828-4610-bedc-0e4fafc54873',
 15: '49d903d1-33ce-4fa1-945f-aae446e15181',
 16: '1a0bfa7e-ed96-43dd-a398-446ea2ebf00f',
 17: 'ad72bda4-642d-4a9d-8041-de28f1c550d6',
 18: '41f9e257-0c9a-457d-b8a2-549daf31730d',
 19: 'fef7ef31-2a06-4891-9bf7-7c828b3be0be',
 20: 'd317e6f9-4ec1-4be9-bf8f-389f619fbd5d',
 21: '1e533866-3a82-4aac-b58b-679c2ef4e97f',
 22: 'cb12b16e-274f-

In [21]:
vector_store.get_by_ids(['8ae5fc30-4839-429c-aae8-04c82ed69aa9'])

[Document(id='8ae5fc30-4839-429c-aae8-04c82ed69aa9', metadata={}, page_content='advance at your ideal pace. There is always something\nnew to discover on Brilliant. Want to better understand optics after watching this video? Well, their "Scientific\nThinking" course is a great place to start. It helps you think like an engineer by showing you how to break down large concepts into smaller,\nmore understandable pieces. Whether you are conquering\nfundamental math, algebra or calculus, diving deep into algorithms, exploring material science or understanding the physics\nthat will take us beyond EUV, Brilliant will help you get there. And if, like me, you\'ve\nresolved to learn more in the New Year, then Brilliant is a great\nway to actually make that resolution stick. So to learn for free, go to brilliant.org/Veritasium, scan the QR code on screen, or click the link in the description. Brilliant has also given\nour viewers 20% off an annual Premium subscription, which gives you unlimited 

## Retrieval

In [22]:
retriever= vector_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [23]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020D835F76B0>, search_kwargs={'k': 4})

In [34]:
docs=retriever.invoke("About ASML")
docs

[Document(id='7d24ebf7-fdcc-4103-8456-07dbc6ade184', metadata={}, page_content="EUV lithography machine. That left just one company, ASML. ASML, which used to stand for Advanced Semiconductor\nMaterials Lithography, is located in a small, nondescript\ntown in the Netherlands. It spun off from Philips back in the '80s with little more than a shed and a barely working\nwafer stepper to its name. But Philips also gave them people, Jos Benschop, ASML's first researcher, and Martin van den Brink, who would eventually become ASML's CTO, and EUV's greatest champion. - And he is really like the\nSteve Jobs of lithography. And he saw EUV coming. - ASML had joined the US\nEUV consortium earlier and now it became their task to find a way to commercialize EUV. They would work together with\ntheir German partner, Zeiss, where Zeiss would take\ncare of the mirrors, and ASML would focus on the light source. One of the first decisions when making any lithography system is"),
 Document(id='23ea85d7-0ae

## Augmentation

In [62]:
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_classic.schema.runnable import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [63]:
# llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task='text-generation',
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
)
model = ChatHuggingFace(llm=llm)

In [50]:
prompt = PromptTemplate(
    template="""
        You are a helpful Assistant.
        Answer only from the given Context.
        If anything not found in context, simply say I don't know.

        {context}
        question:{ques}
    """,
    input_variables=['context','ques']
)

In [51]:
def concatenator(docs):
    return "".join(doc.page_content for doc in docs)

In [52]:
parser=StrOutputParser()

In [53]:
parallel = RunnableParallel({
    "context" : retriever | RunnableLambda(concatenator) | parser,
    "ques" : parser
})

In [54]:
final_prompt = parallel | prompt 

In [61]:
RAG_pipe= final_prompt | llm | parser

In [66]:
RAG_pipe.invoke("What is purpose of EUV?")

'EUV (Extreme Ultraviolet) is used in lithography machines to print smaller features, particularly for making chips, and to achieve this at a manufacturing level.'